# Day 2 — Types and values

> ⚠️ **Why this matters.** Half of all Python bugs are type confusion: you thought it was a string, it was a number; you thought it was a list, it was `None`. Type hints make those bugs visible before you run the code. Today you learn the types and start typing everything, like a senior engineer would.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jakkzz/prince-curriculum/blob/main/phase-1-python-cli/lessons/02-types-and-values.ipynb)

## What you'll do today

**Time:** 90 minutes.

By the end:

- [ ] You can list Python's core built-in types and what each is for
- [ ] You can use type hints fluently — every variable, every parameter, every return
- [ ] You understand `Optional[X]` / `X | None`
- [ ] You've run mypy against typed code and broken it on purpose
- [ ] You know when types are checked (statically) vs not (at runtime)

## The mental model

Every value in Python has a **type**. The type determines what you can do with it.

```
42        → int      (can do math)
3.14      → float    (can do math, imprecise)
"hello"   → str      (can slice, upper, lower)
True      → bool     (can do logic)
[1, 2, 3] → list     (can iterate, append)
{1, 2, 3} → set      (fast 'in' lookup)
{"a": 1}  → dict     (key-value)
None      → NoneType (the absence of a value)
```

**Python is dynamically typed** — you don't declare types up front. **Python supports type hints** — you can declare them anyway, for documentation and bug-catching. Modern Python code does declare them.

> 💡 **In the wild:** Every Python codebase at Anthropic, Stripe, Meta, Google, Microsoft — all heavily typed. `mypy --strict` is the bar. Code without types is treated as legacy.

## 1. The built-in types

`type()` tells you what something is.

In [1]:
type(42)

int

In [2]:
type("hello")

str

In [3]:
type([1, 2, 3])

list

In [4]:
type(None)

NoneType

## 2. Type hints — your code's self-documentation

A type hint after `:` on a variable declares what type it should be.

In [5]:
name: str = "Prince"
age: int = 16
height_m: float = 1.72
is_student: bool = True
favorite_numbers: list[int] = [7, 13, 42]

Type hints are **not enforced at runtime**. Python won't crash if you assign a string to a variable hinted as `int`. So why bother?

1. **Tools check them.** `mypy` (and your IDE) catches mistakes before you run.
2. **Self-documentation.** Anyone reading the code (including you, in 3 months) knows what's expected.
3. **Autocomplete.** Your editor knows what methods are available because it knows the type.

Run this — it doesn't crash, even though we lied about the type:

In [6]:
x: int = "not actually an int"   # mypy would complain, runtime doesn't care
print("actual type:", type(x))

actual type: <class 'str'>


> ⚠️ **Type hints are honor-system at runtime.** Run mypy as part of your workflow or they're decoration. We'll set up `mypy --strict` next.

## 3. Collections — list, tuple, set, dict

Four ways to hold multiple values. Each optimized for different patterns.

| Type | Syntax | Mutable? | Ordered? | Use when |
|------|--------|----------|----------|----------|
| `list` | `[1, 2, 3]` | yes | yes | You'll append and iterate |
| `tuple` | `(1, 2, 3)` | no | yes | Fixed group of related values |
| `set` | `{1, 2, 3}` | yes | no | You need fast `in` checks, unique values |
| `dict` | `{"a": 1}` | yes | yes (since 3.7) | Key-value mapping |

In [7]:
numbers: list[int] = [7, 13, 42]
numbers.append(99)
numbers

[7, 13, 42, 99]

In [8]:
unique: set[int] = {1, 2, 2, 3, 3, 4}   # duplicates dropped automatically
unique

{1, 2, 3, 4}

In [9]:
3 in unique     # set lookup is O(1) — instant even for millions of items

True

In [10]:
user: dict[str, int | str] = {"name": "Prince", "age": 16}
user["age"]

16

## 4. `None` and `Optional` / `| None`

`None` is the absence of a value. Different from `0`, `""`, `False`, `[]` — those are *values*. `None` is *no value*.

When something might be `None`, the type is `X | None` (modern) or `Optional[X]` (older). Both mean the same thing.

In [11]:
def greet(name: str | None) -> str:
    if name is None:
        return "Hello, stranger"
    return f"Hello, {name}"

greet("Prince")

'Hello, Prince'

In [12]:
greet(None)

'Hello, stranger'

> 💡 **Always check with `is None`, not `== None`.** The `is` operator checks identity (same object); `== None` calls `.__eq__()` which can be overridden in weird ways. `is None` is the idiom.

## 5. Conversions — when types meet

Python doesn't auto-convert (unlike JavaScript). You have to be explicit.

| From | To | Function |
|------|----|----------|
| any | str | `str(x)` |
| str | int | `int("42")` |
| str | float | `float("3.14")` |
| int/float | bool | `bool(x)` — 0/0.0/empty → False, else True |
| list | tuple / set | `tuple(xs)` / `set(xs)` |

In [13]:
age = 16
"I am " + str(age) + " years old."   # must convert int to str to concatenate

'I am 16 years old.'

In [14]:
user_input = "42"            # input() always returns str
doubled = int(user_input) * 2
doubled

84

## 6. Truthiness — when types are used as booleans

Python has a concept of "truthy" and "falsy." In an `if` or `while`, certain values count as `False` even when they're not actually `False`.

In [15]:
for value in [0, 1, "", "hello", [], [0], None]:
    print(f"{value!r} → {bool(value)}")

0 → False
1 → True
'' → False
'hello' → True
[] → False
[0] → True
None → False


**Falsy:** `False`, `None`, `0`, `0.0`, `""`, `[]`, `{}`, `set()`.

**Everything else** is truthy. So `if name:` works, but be careful — it'll also be False if `name == ""`. If you mean "name is None", say `if name is None:`.

## 7. End-of-day mini-project — `word_list.py`

> 🎯 **Today's piece of [English Helper](RUNNING-PROJECT.md):** a typed list of vocabulary words with their IPA pronunciation. Day 1 you printed one card. Today you handle many.

### What you're building

A Python script that prints a list of English words with their International Phonetic Alphabet (IPA) pronunciation and Thai translation, neatly aligned:

```
Word                 IPA                       Thai
--------------------------------------------------------------------
ubiquitous           /juːˈbɪkwɪtəs/            พบเห็นได้ทั่วไป
thorough             /ˈθʌrə/                   ละเอียด
inevitable           /ɪˈnevɪtəbl/              ที่ไม่อาจหลีกเลี่ยงได้
nuance               /ˈnuːɑːns/                ความแตกต่างที่ละเอียดอ่อน
resilient            /rɪˈzɪliənt/              ยืดหยุ่นสามารถฟื้นตัวได้
```

### Requirements

- Store words as a `list[dict[str, str]]` — list of dicts.
- Each dict has keys `"word"`, `"ipa"`, `"thai"`.
- Use a `for` loop to print each row.
- Columns aligned with f-string padding: `f"{word:20} {ipa:25} {thai}"`.
- Type hints on every variable.
- File passes `uv run mypy word_list.py` with no errors.

### IPA — what is it?

The **International Phonetic Alphabet** is a universal way to write pronunciation. `/θ/` is the "th" in *thorough*; `/ʌ/` is the "u" in *cup*. Dictionaries use IPA so the same word reads the same in every language.

Look up IPA for your words at:
- [dictionary.cambridge.org](https://dictionary.cambridge.org/) — search a word, see IPA next to it
- [Wiktionary](https://en.wiktionary.org/) — usually shows both UK and US pronunciations

### Try it

In [ ]:
# Write your code here. Try it in this cell first, then copy to word_list.py.

# Your code here


<details>
<summary>Solution — try first!</summary>

```python
# word_list.py
WordEntry = dict[str, str]

words: list[WordEntry] = [
    {"word": "ubiquitous",  "ipa": "/juːˈbɪkwɪtəs/", "thai": "พบเห็นได้ทั่วไป"},
    {"word": "thorough",    "ipa": "/ˈθʌrə/",        "thai": "ละเอียด"},
    {"word": "inevitable",  "ipa": "/ɪˈnevɪtəbl/",   "thai": "ที่ไม่อาจหลีกเลี่ยงได้"},
    {"word": "nuance",      "ipa": "/ˈnuːɑːns/",     "thai": "ความแตกต่างที่ละเอียดอ่อน"},
    {"word": "resilient",   "ipa": "/rɪˈzɪliənt/",   "thai": "ยืดหยุ่นสามารถฟื้นตัวได้"},
]

# Header
print(f"{'Word':20} {'IPA':25} Thai")
print("-" * 68)

# Rows
for entry in words:
    print(f"{entry['word']:20} {entry['ipa']:25} {entry['thai']}")

# Stats
print()
print(f"Total words in your vocabulary: {len(words)}")
```

**Run it:**

```bash
cd ~/prince/scratch/english-helper
uv run mypy word_list.py    # should say: Success
uv run python word_list.py
```

**Stretch (try after the basic version works):**
- Add 5 more words you actually want to learn.
- Add a `pos` (part of speech) column.
- Use `sorted()` to print alphabetically.
- Count: how many start with a vowel? (`sum(1 for w in words if w['word'][0] in 'aeiou')`)
</details>

## 8. Run mypy and break it on purpose

Type checkers catch bugs *before you run the code*. That's the magic.

In your terminal:

```bash
cd ~/prince/scratch/english-helper
uv run mypy word_list.py
```

If it says `Success: no issues found` — you're golden.

**Now break it.** Edit `word_list.py` and add this line right after the list:

```python
words.append({"word": "epiphany", "ipa": 42, "thai": "การตรัสรู้"})
```

(IPA is an int, not a str — that's wrong.) Run mypy again:

```
word_list.py:11: error: Dict entry 1 has incompatible type "str": "int"; expected "str": "str"  [dict-item]
Found 1 error in 1 file (checked 1 source file)
```

**mypy caught a real bug.** Without types, you'd hit this at runtime — maybe in production. Now fix it (use a real IPA string) and re-run mypy.

> 🎯 **Make this your habit.** From now on, every commit you make in Phase 1 should pass: `ruff check . && ruff format . && mypy .`. Set an alias for it.

## Connect to the project

> 🎯 **Connects to the project:** `word_list.py` is the second piece of English Helper. Tomorrow (Day 3 — control flow) you'll learn `if`, `for`, `while`, and `input()` — and you'll turn this static word list into an **interactive pronunciation quiz**. The data structure you built today is the data the quiz draws from.

## Self-check

<details>
<summary>1. What's the difference between <code>list</code> and <code>tuple</code>?</summary>

Both are ordered sequences. `list` is mutable (you can add/remove/change). `tuple` is immutable (fixed once created). Use tuple for fixed, related groups; list for collections you'll modify.
</details>

<details>
<summary>2. Why are type hints not enforced at runtime?</summary>

Python prioritizes flexibility — type hints are a layer on top, optional. Enforcement is done by tools (mypy, your IDE) statically. This means types catch bugs at edit/check time, not at run time. It also means you have to actually run mypy for them to matter.
</details>

<details>
<summary>3. What does <code>list[int] | None</code> mean as a type?</summary>

Either a list of integers, OR `None`. Useful when a function might "return the list of results, or nothing."
</details>

<details>
<summary>4. Why use <code>is None</code> instead of <code>== None</code>?</summary>

`is` checks identity ("same object in memory"). `==` calls `.__eq__()`, which a custom class can override to lie. `None` is a singleton, so `is None` is unambiguous, fast, and idiomatic.
</details>

<details>
<summary>5. What's "truthy" mean? Give 3 falsy values that aren't <code>False</code>.</summary>

Truthy = a value that evaluates to `True` in an `if`/`while`. Falsy values include: `False`, `None`, `0`, `0.0`, `""` (empty string), `[]` (empty list), `{}` (empty dict), `set()` (empty set).
</details>

## What's next

Tomorrow: **control flow** — `if`, `for`, `while`, comprehensions. You'll turn `word_list.py` into an interactive quiz.

**Don't forget the quiz!** Open [02-types-and-values-quiz.ipynb](02-types-and-values-quiz.ipynb) for 30+ questions to confirm you've absorbed today's lesson.